# QC A3 Parity Pilot

This notebook validates the Investintell A3 path inside QuantConnect Research without using QC FRED or returns as an A3 objective. It consumes immutable Investintell L2/revision-uncertainty/config artifacts and compares normalized outputs against the local worker bundle.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from qc_a3_core import A3ParityConfig, run_parity


## Object Store contract

Upload every file listed in `object_store_manifest.json` to its matching Object Store key. The core A3 run must read only these Investintell artifacts. QC History/FRED is intentionally absent from this notebook.

In [ ]:
manifest_path = Path("object_store_manifest.json")
if not manifest_path.exists():
    manifest_path = Path("_tmp_qc_a3_parity/object_store_manifest.json")

manifest = json.loads(manifest_path.read_text())
assert manifest["bridge_scope"] == "qc_research_parity_only"
assert manifest["runtime_activation"] is False
assert manifest["a3_status"] == "open_macro_v03"
manifest["selected"]


In [ ]:
# Local fallback paths from the exported bundle. In QC Research, replace these
# with qb.object_store.get_file_path(manifest['object_store_keys'][...]).
bundle_dir = manifest_path.parent
config = A3ParityConfig(
    feature_manifest=bundle_dir / "manifests" / "feature_manifest.json",
    revision_uncertainty_manifest=bundle_dir / "manifests" / "revision_uncertainty_manifest.json",
    config_catalog=bundle_dir / "manifests" / "config_catalog.normalized.json",
    a32_grid_dir=bundle_dir / "manifests",
    output_dir=bundle_dir / "qc_research_parity_run",
    a31_name=manifest["selected"]["a31_config_name"],
    a32_name=manifest["selected"]["a32_config_name"],
    worker_commit=manifest["worker_commit"],
)
report = run_parity(config)
report["comparison"]


In [ ]:
assert report["runtime_activation"] is False
assert report["a4_status"] == "harness_ready_provisional_A3"
assert report["a5_status"] == "blocked"
print(report["runtime_replay_logical_hash"])
print(report["metric_rows_logical_hash"])
